# RoadEye – AI-Based Road Risk Predictor  
This notebook contains the ML pipeline for RoadEye.  
Dataset includes features like Weather, Time, Road Type, Visibility, Accident History etc.  
Model used: Random Forest Classifier.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!ls "/content/drive/MyDrive/LochanSaini_RoadEye_I_2025"


 Dataset  'Documentation '   Model   Notes  'UI Dashboard '


!ls "/content/drive/MyDrive"

In [3]:
!ls -la "/content/drive/MyDrive/LochanSaini_RoadEye_I_2025/Dataset"

total 1323
-rw------- 1 root root   40841 Feb 22 05:00 expanded_dataset.csv
-rw------- 1 root root     195 Mar 23 07:36 expanded_dataset.gsheet
-rw------- 1 root root     195 Dec  7 02:50 RoadEye_Dataset.csv.gsheet
-rw------- 1 root root 1312091 Mar 23 07:37 roadeye_dataset_v3.csv
-rw------- 1 root root     195 Feb 22 03:24 RoadEye_Segments_Jaipur.csv.gsheet


In [4]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/LochanSaini_RoadEye_I_2025/Dataset/roadeye_dataset_v3.csv')
print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head()

Shape: (10008, 22)
Columns: ['segment_id', 'city', 'road_name', 'segment_name', 'start_latitude', 'start_longitude', 'end_latitude', 'end_longitude', 'road_type', 'speed_limit_est(inKm/h)', 'blackspot_flag', 'road_surface', 'season', 'season_name', 'time_of_day', 'time_name', 'weather_type', 'weather_name', 'traffic_density', 'traffic_name', 'road_surface_name', 'risk_label']


,segment_id,city,road_name,segment_name,start_latitude,start_longitude,end_latitude,end_longitude,road_type,speed_limit_est(inKm/h),...,season,season_name,time_of_day,time_name,weather_type,weather_name,traffic_density,traffic_name,road_surface_name,risk_label
0,NH48_01,Jaipur,NH-48,NH-48 Aspura Stretch,27.454846,76.030283,27.456285,76.033271,0,90,...,0,Summer,0,Night,0,Clear,0,Low,Poor,2
1,NH48_01,Jaipur,NH-48,NH-48 Aspura Stretch,27.454846,76.030283,27.456285,76.033271,0,90,...,0,Summer,0,Night,0,Clear,1,Medium,Poor,2
2,NH48_01,Jaipur,NH-48,NH-48 Aspura Stretch,27.454846,76.030283,27.456285,76.033271,0,90,...,0,Summer,0,Night,0,Clear,2,High,Poor,2
3,NH48_01,Jaipur,NH-48,NH-48 Aspura Stretch,27.454846,76.030283,27.456285,76.033271,0,90,...,0,Summer,1,Peak Hours,0,Clear,0,Low,Poor,1
4,NH48_01,Jaipur,NH-48,NH-48 Aspura Stretch,27.454846,76.030283,27.456285,76.033271,0,90,...,0,Summer,1,Peak Hours,0,Clear,1,Medium,Poor,1


In [5]:
# Features jo model use karega
features = [
    'start_latitude', 'start_longitude',
    'end_latitude', 'end_longitude',
    'road_type', 'speed_limit_est(inKm/h)',
    'blackspot_flag', 'road_surface',
    'season', 'time_of_day',
    'weather_type', 'traffic_density'
]

X = df[features]
y = df['risk_label']

print("X shape:", X.shape)
print("y distribution:")
print(y.value_counts())

X shape: (10008, 12)
y distribution:
risk_label
1    5043
2    2961
0    2004
Name: count, dtype: int64


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

model = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    max_depth=15,
    random_state=42
)
model.fit(X_train, y_train)
print("Model trained successfully ✅")

Model trained successfully ✅


In [7]:
print(y.value_counts())

risk_label
1    5043
2    2961
0    2004
Name: count, dtype: int64


In [8]:
from sklearn.metrics import accuracy_score, classification_report

predictions = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, predictions))
print()
print(classification_report(
    y_test, predictions,
    target_names=['Low Risk', 'Medium Risk', 'High Risk']
))

Accuracy: 0.971028971028971

              precision    recall  f1-score   support

    Low Risk       0.99      0.95      0.97       401
 Medium Risk       0.97      0.98      0.97      1009
   High Risk       0.97      0.98      0.97       592

    accuracy                           0.97      2002
   macro avg       0.97      0.97      0.97      2002
weighted avg       0.97      0.97      0.97      2002



In [9]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, X, y, cv=5)
print("Cross-val scores:", scores)
print("Mean accuracy:", scores.mean().round(4))

Cross-val scores: [0.87862138 0.77172827 0.6998002  0.7946027  0.92753623]
Mean accuracy: 0.8145


In [10]:
import numpy as np

importances = model.feature_importances_
indices = np.argsort(importances)[::-1]

print("Feature Importances (ranked):")
for i in range(len(features)):
    print(f"  {features[indices[i]]:35s} {importances[indices[i]]:.4f}")

Feature Importances (ranked):
  weather_type                        0.2088
  blackspot_flag                      0.1598
  speed_limit_est(inKm/h)             0.1382
  time_of_day                         0.1366
  road_type                           0.0873
  traffic_density                     0.0868
  road_surface                        0.0740
  season                              0.0414
  end_longitude                       0.0178
  start_longitude                     0.0171
  start_latitude                      0.0162
  end_latitude                        0.0161


In [11]:
import joblib
import os

os.makedirs("/content/drive/MyDrive/LochanSaini_RoadEye_I_2025/Model", exist_ok=True)

joblib.dump(model, "/content/drive/MyDrive/LochanSaini_RoadEye_I_2025/Model/model.pkl")
print("Model saved ✅")
print("Features used:", features)

Model saved ✅
Features used: ['start_latitude', 'start_longitude', 'end_latitude', 'end_longitude', 'road_type', 'speed_limit_est(inKm/h)', 'blackspot_flag', 'road_surface', 'season', 'time_of_day', 'weather_type', 'traffic_density']
